### try

In [18]:
import pandas as pd

# 1. Load data
df = pd.read_csv(
    'transactions_train.csv',
    usecols=['t_dat', 'article_id', 'price'],
    parse_dates=['t_dat']
)

print("Data loaded. Grouping and Pivoting...")

# 2. Group by Article and Week
# We aggregate to get the Count and Mean Price.
weekly_data = df.groupby(
    ['article_id', pd.Grouper(key='t_dat', freq='W')]
).agg(
    count=('article_id', 'size'),
    mean_price=('price', 'mean')
)

# 3. Unstack to move weeks to columns
matrix = weekly_data.unstack()

# 4. Handle Missing Values
# Fill only the 'count' columns with 0. 
# We leave mean_price as NaN because "0.00" average price is misleading for weeks with no sales.
matrix['count'] = matrix['count'].fillna(0)

# 5. Clean up the Column Names
# Swap levels to get (Date, Metric) order so we can keep weeks together.
matrix.columns = matrix.columns.swaplevel(0, 1)
matrix.sort_index(axis=1, level=0, inplace=True)

# Flatten columns to strings like '2018-09-23_count', '2018-09-23_mean_price'
matrix.columns = [
    f"{date.strftime('%Y-%m-%d')}_{metric}" 
    for date, metric in matrix.columns
]

# 6. Save to CSV
output_file = 'all_articles_weekly_mean_stats_test.csv'
matrix.to_csv(output_file)

print(f"Matrix saved to '{output_file}'.")
print(f"Rows (Articles): {matrix.shape[0]}")
print(f"Columns: {matrix.shape[1]}")
print("\nPreview of the data:")
print(matrix.head())

Data loaded. Grouping and Pivoting...
Matrix saved to 'all_articles_weekly_mean_stats_test.csv'.
Rows (Articles): 104547
Columns: 212

Preview of the data:
            2018-09-23_count  2018-09-23_mean_price  2018-09-30_count  \
article_id                                                              
108775015              115.0               0.008337             547.0   
108775044               57.0               0.008361             226.0   
108775051               21.0               0.005002              38.0   
110065001               13.0               0.024820              56.0   
110065002               11.0               0.024444              16.0   

            2018-09-30_mean_price  2018-10-07_count  2018-10-07_mean_price  \
article_id                                                                   
108775015                0.007961             315.0               0.008343   
108775044                0.007887             139.0               0.008311   
108775051           

In [ ]:
import pandas as pd

df = pd.read_csv('all_articles_weekly_mean_stats_test.csv')
#print(df.head())
#print(df.columns.tolist())

import numpy as np

# 1. Identify price columns
price_cols = [col for col in df.columns if 'mean_price' in col]

# 2. Compute row means
# We use numeric_only=True explicitly, though these cols are numeric.
row_means = df[price_cols].mean(axis=1)

# 3. Create mask for outliers
# We need to handle the comparison row-wise.
# Comparing a DataFrame with a Series (axis=0 of the Series matches axis=0 of DataFrame) requires specific alignment.
# df[price_cols] is (n_rows, n_weeks)
# row_means is (n_rows,)
# We want to broadcast row_means across columns.
# df.lt(series, axis=0) works.

lower_bound = row_means * 0.8
upper_bound = row_means * 1.2

# Mask where price is OUTSIDE the bounds
# Note: NaN comparisons will be False, so NaNs are preserved (not removed).
mask_lower = df[price_cols].lt(lower_bound, axis=0)
mask_upper = df[price_cols].gt(upper_bound, axis=0)
outlier_mask = mask_lower | mask_upper

# 4. Set outliers to NaN
# We need to iterate because we need to set the corresponding count column too.
# Or we can do it smartly.
# Let's iterate over the price columns to be safe and clear.

# Create a copy to modify
df_filtered = df.copy()

for col in price_cols:
    # Identify rows where this column is an outlier
    is_outlier = outlier_mask[col]
    
    if is_outlier.sum() > 0:
        # Set price to NaN
        df_filtered.loc[is_outlier, col] = np.nan
        
        # Set corresponding count to NaN
        count_col = col.replace('_mean_price', '_count')
        if count_col in df.columns:
            df_filtered.loc[is_outlier, count_col] = np.nan

# Check some stats to see if it worked
print("Original non-null values in price cols:", df[price_cols].notna().sum().sum())
print("Filtered non-null values in price cols:", df_filtered[price_cols].notna().sum().sum())

# Save
df_filtered.to_csv('filtered_weekly_mean_stats_test.csv', index=False)

   article_id  2018-09-23_count  2018-09-23_mean_price  2018-09-30_count  \
0   108775015             115.0               0.008337             547.0   
1   108775044              57.0               0.008361             226.0   
2   108775051              21.0               0.005002              38.0   
3   110065001              13.0               0.024820              56.0   
4   110065002              11.0               0.024444              16.0   

   2018-09-30_mean_price  2018-10-07_count  2018-10-07_mean_price  \
0               0.007961             315.0               0.008343   
1               0.007887             139.0               0.008311   
2               0.005068              45.0               0.005034   
3               0.023722              37.0               0.024229   
4               0.023502              16.0               0.024136   

   2018-10-14_count  2018-10-14_mean_price  2018-10-21_count  ...  \
0             360.0               0.007627             259.

In [21]:
import pandas as pd

# Load the previously filtered data
df_filtered = pd.read_csv('filtered_weekly_mean_stats_test.csv')

# Identify columns to keep
# We want 'article_id' and any column containing '_count'
cols_to_keep = ['article_id'] + [col for col in df_filtered.columns if '_count' in col]

# Select those columns
df_counts_only = df_filtered[cols_to_keep]

# Inspect
print(df_counts_only.head())
print(df_counts_only.columns.tolist())

# Save to CSV
df_counts_only.to_csv('filtered_weekly_counts_test.csv', index=False)

   article_id  2018-09-23_count  2018-09-30_count  2018-10-07_count  \
0   108775015             115.0             547.0             315.0   
1   108775044              57.0             226.0             139.0   
2   108775051              21.0              38.0              45.0   
3   110065001               NaN               NaN               NaN   
4   110065002               NaN               NaN               NaN   

   2018-10-14_count  2018-10-21_count  2018-10-28_count  2018-11-04_count  \
0             360.0             259.0             434.0             373.0   
1             155.0             115.0             160.0             107.0   
2              44.0              24.0              19.0               5.0   
3               NaN               NaN               NaN               NaN   
4              20.0               NaN               NaN               NaN   

   2018-11-11_count  2018-11-18_count  ...  2020-07-26_count  \
0             312.0             244.0  ...    

In [22]:
import pandas as pd

# 1. Load the weekly sales data (created in previous steps)
weekly_sales_df = pd.read_csv('filtered_weekly_counts_test.csv')

# 2. Define the specific columns you want from the articles file
article_cols = [
    'article_id', 
    'prod_name', 
    'product_type_name', 
    'graphical_appearance_name', 
    'colour_group_name', 
    'perceived_colour_value_name', 
    'perceived_colour_master_name', 
    'department_name', 
    'index_name', 
    'index_group_name', 
    'section_name', 
    'garment_group_name', 
    'detail_desc'
]

# 3. Load the articles data
# We use 'usecols' to save memory by loading only what we need
articles_df = pd.read_csv('articles.csv', usecols=article_cols)

# 4. Merge the dataframes
# how='left' ensures we keep all the sales rows, even if an article is missing details (unlikely)
print("Merging data...")
merged_df = pd.merge(weekly_sales_df, articles_df, on='article_id', how='left')

# 5. Save the final report
output_file = 'weekly_sales_with_details_filtered_test.csv'
merged_df.to_csv(output_file, index=False)

print(f"Successfully merged. Saved to '{output_file}'.")
print(merged_df.head())

Merging data...
Successfully merged. Saved to 'weekly_sales_with_details_filtered_test.csv'.
   article_id  2018-09-23_count  2018-09-30_count  2018-10-07_count  \
0   108775015             115.0             547.0             315.0   
1   108775044              57.0             226.0             139.0   
2   108775051              21.0              38.0              45.0   
3   110065001               NaN               NaN               NaN   
4   110065002               NaN               NaN               NaN   

   2018-10-14_count  2018-10-21_count  2018-10-28_count  2018-11-04_count  \
0             360.0             259.0             434.0             373.0   
1             155.0             115.0             160.0             107.0   
2              44.0              24.0              19.0               5.0   
3               NaN               NaN               NaN               NaN   
4              20.0               NaN               NaN               NaN   

   2018-11-11_cou

In [38]:
import pandas as pd

# Load the original dataset
df = pd.read_csv('weekly_sales_with_details_filtered_test.csv')

# Get a list of all column names to find the indices for slicing
all_cols = df.columns.tolist()

# Define the start and end date columns for the sub-period
start_date_col = '2019-03-03_count'
end_date_col = '2019-09-22_count'

# Identify the descriptive columns (everything after the final date column)
last_date_col = '2020-09-27_count'
last_date_idx = all_cols.index(last_date_col)
desc_cols = all_cols[last_date_idx + 1:]

# Combine 'article_id', the selected date range, and the metadata columns
selected_cols = (
    ['article_id'] + 
    all_cols[all_cols.index(start_date_col):all_cols.index(end_date_col)+1] + 
    desc_cols
)

# Filter the DataFrame to the selected columns
sub_df = df[selected_cols]

# Save the subset to a new CSV file
sub_df.to_csv('weekly_sales_with_details_filtered_subperiod_test.csv', index=False)

# Display the first few rows to verify
print(sub_df.head())

   article_id  2019-03-03_count  2019-03-10_count  2019-03-17_count  \
0   108775015             239.0             244.0             248.0   
1   108775044             175.0             149.0             138.0   
2   108775051               0.0               0.0               0.0   
3   110065001               NaN               NaN               NaN   
4   110065002              10.0               9.0               3.0   

   2019-03-24_count  2019-03-31_count  2019-04-07_count  2019-04-14_count  \
0             258.0             285.0             353.0             206.0   
1             156.0             182.0             312.0             207.0   
2               0.0               0.0               0.0               0.0   
3               NaN               NaN               NaN               NaN   
4               4.0               4.0               NaN               NaN   

   2019-04-21_count  2019-04-28_count  ...  graphical_appearance_name  \
0             183.0             314.0

In [34]:
import pandas as pd

# Load the dataset
df = pd.read_csv('weekly_sales_with_details_filtered_subperiod.csv')

# Identify count columns (assuming they end with '_count')
count_cols = [col for col in df.columns if '_count' in col]

# List to store summary data
summary_data = []

# List to store valid indices for the cleaned dataset
valid_indices = []

# Get unique product types
product_types = df['product_type_name'].unique()

for pt in product_types:
    # Get subset for the current product type
    sub_df = df[df['product_type_name'] == pt]
    
    # Filter: Drop rows with any NaN in count columns
    # We use a temporary dataframe to hold the non-NA rows first
    non_na_df = sub_df.dropna(subset=count_cols)
    
    # Filter: Drop rows with any 0.0 in count columns
    # We create a boolean mask where True means all count columns are non-zero
    mask_non_zero = (non_na_df[count_cols] != 0).all(axis=1)
    
    # Apply the mask to get the final valid rows for this product type
    clean_sub_df = non_na_df[mask_non_zero]
    
    # Store summary info
    summary_data.append({
        'product_type_name': pt,
        'original_count': len(sub_df),
        'remaining_count': len(clean_sub_df)
    })
    
    # Store indices of valid rows to reconstruct the full cleaned dataframe later
    valid_indices.extend(clean_sub_df.index.tolist())

# Create summary DataFrame
summary_df = pd.DataFrame(summary_data)
# Sort by remaining_count descending for better visibility
summary_df = summary_df.sort_values(by='remaining_count', ascending=False)

# Save summary to CSV
summary_df.to_csv('product_type_summary_test.csv', index=False)

# Create the final cleaned DataFrame using the collected valid indices
cleaned_df = df.loc[valid_indices]

# Save cleaned data to CSV
cleaned_df.to_csv('cleaned_product_data_test.csv', index=False)

In [35]:
import pandas as pd

# Define file paths
input_file = 'weekly_sales_with_details_filtered_subperiod.csv'
output_file = 'weekly_sales_filtered_zeros_and_10nas_test.csv'

# Load the dataframe
df = pd.read_csv(input_file)

# Identify columns with sales counts (columns ending in '_count')
count_cols = [col for col in df.columns if '_count' in col]

# Step 1: Remove items with any zero (excluding NAs)
# We check if any value in the count columns is equal to 0 for each row.
rows_with_zeros = (df[count_cols] == 0).any(axis=1)
df_filtered_1 = df[~rows_with_zeros].copy()

# Step 2: Remove items with more than 10 NAs
# We count the number of missing values (NaNs) in the count columns for each row.
na_counts = df_filtered_1[count_cols].isna().sum(axis=1)
df_filtered_2 = df_filtered_1[na_counts <= 10].copy()

# Save the result to a CSV file
df_filtered_2.to_csv(output_file, index=False)

print(f"Processed data saved to {output_file}")
print(f"Final number of rows: {len(df_filtered_2)}")

Processed data saved to weekly_sales_filtered_zeros_and_10nas_test.csv
Final number of rows: 1719


In [36]:
import pandas as pd

# Load the previously filtered dataframe
input_file = 'weekly_sales_filtered_zeros_and_10nas_test.csv'
df = pd.read_csv(input_file)

# Count the frequency of each product type
type_counts = df['product_type_name'].value_counts()

# Identify types that have more than 20 items
types_to_keep = type_counts[type_counts >= 20].index

# Filter the dataframe to only include those types
df_filtered_types = df[df['product_type_name'].isin(types_to_keep)].copy()

# Save the result to a new CSV file
output_file = 'weekly_sales_filtered_zeros_10nas_min20types_test.csv'
df_filtered_types.to_csv(output_file, index=False)

print(f"Filtered data saved to {output_file}")
print(f"Final number of rows: {len(df_filtered_types)}")

Filtered data saved to weekly_sales_filtered_zeros_10nas_min20types_test.csv
Final number of rows: 1469
